# Clase 12 — Visión y lenguaje: proyecto final y cierre del curso

## Objetivos de aprendizaje

- Entender transfer learning con VGG16 (reutilizar un modelo preentrenado en ImageNet para un problema nuevo con pocos datos).
- Entender cómo BLIP genera descripciones de imágenes en lenguaje natural.
- Ejecutar BLIP de verdad, gratis, sobre una imagen real descargada por URL.
- Repasar, en la última celda, todo lo que construiste a lo largo de las 12 clases del curso.

## Qué vamos a construir

La arquitectura conceptual de un clasificador de daños en aeronaves con VGG16 (transfer learning), y un generador de descripciones de imágenes real y ejecutable con BLIP.

## Repaso teórico: transfer learning con VGG16

El caso de negocio real de este proyecto (material B.8 del curso, `Aircraft Damage Dataset v1`) es automatizar la inspección de daños en aviones: clasificar fotos como abolladura (*dent*) o grieta (*crack*).

En vez de entrenar una CNN desde cero (que necesitaría muchísimos datos), reutilizamos **VGG16**, una red convolucional ya entrenada en ImageNet (más de un millón de imágenes, mil categorías). La estrategia:

1. Cargar VGG16 sin su clasificador final: `VGG16(weights='imagenet', include_top=False)`.
2. Congelar sus capas (`layer.trainable = False`) para no destruir lo que ya aprendió.
3. Añadir un clasificador propio encima: `Flatten()` → `Dense(512, relu)` → `Dropout(0.3)` → `Dense(512, relu)` → `Dropout(0.3)` → `Dense(1, sigmoid)`.
4. Entrenar **solo** esas capas nuevas con las fotos de dent/crack.

Como el dataset completo de aviones es pesado para una sesión de clase, aquí vas a ver la arquitectura funcionando de verdad (VGG16 se descarga solo, sin API key, viene con Keras/TensorFlow), sin descargar el dataset completo de fotos de aviones.

In [ ]:
# VGG16 viene incluido con TensorFlow/Keras: no hace falta ninguna clave de API,
# solo se descargan los pesos preentrenados en ImageNet la primera vez que se usa.
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers

# Cargamos VGG16 sin su clasificador final, con el tamano de imagen estandar (224x224).
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Congelamos todas las capas del modelo base: no queremos que el entrenamiento
# posterior destruya el conocimiento que VGG16 ya aprendio de ImageNet.
for layer in base_model.layers:
    layer.trainable = False

# Anadimos nuestro propio clasificador encima del modelo congelado
# (para el caso real: clasificacion binaria dent vs crack).
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    loss="binary_crossentropy",
    optimizer=optimizers.Adam(learning_rate=0.0001),
    metrics=["accuracy"],
)

model.summary()

Fíjate en el resumen (`model.summary()`): la parte de `vgg16` tiene millones de parámetros, pero todos están **congelados** (no entrenables) — solo las capas `Dense` que añadiste al final son las que se entrenarían con tus propias imágenes de dent/crack. Eso es transfer learning: reutilizas el "saber ver" genérico de VGG16 y solo entrenas la parte específica de tu problema.

En un proyecto real, el siguiente paso sería:

```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    "aircraft_damage_dataset/train",   # carpetas train/dent y train/crack
    target_size=(224, 224),
    class_mode="binary",
)

model.fit(train_generator, validation_data=valid_generator, epochs=10)
```

No lo ejecutamos aquí porque requiere descargar el dataset completo de imágenes de aviones — pero la arquitectura de arriba es exactamente la que usarías.

## Repaso teórico: BLIP, un modelo que describe imágenes

**BLIP** (*Bootstrapping Language-Image Pre-training*) es un modelo multimodal: recibe una imagen y genera una descripción en lenguaje natural. En el proyecto original se usa dos veces con dos prompts distintos:

- `"This is a picture of"` → descripción breve.
- `"This is a detailed photo showing"` → descripción más detallada.

A diferencia de VGG16 (que clasifica entre categorías fijas), BLIP genera texto libre — responde "¿qué hay en esta imagen?", no "¿a qué categoría pertenece?". Vamos a ejecutarlo de verdad, sobre una imagen pública descargada por URL.

In [ ]:
# Instalamos lo necesario (gratuito, sin API key).
!pip install -q transformers pillow requests

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from transformers import BlipProcessor, BlipForConditionalGeneration

# Cargamos BLIP: procesador (prepara imagen+texto) y modelo (genera la descripcion).
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Descargamos una imagen publica de ejemplo (foto de un perro en una playa,
# una imagen de muestra clasica de la documentacion de Hugging Face/BLIP).
url = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
respuesta = requests.get(url)
imagen = Image.open(BytesIO(respuesta.content)).convert("RGB")

imagen

In [ ]:
# Descripcion breve: el prompt corto empuja al modelo a completar la frase.
inputs = processor(imagen, "a picture of", return_tensors="pt")
salida = blip_model.generate(**inputs, max_new_tokens=30)
caption = processor.decode(salida[0], skip_special_tokens=True)
print("Descripcion breve:", caption)

# Descripcion detallada: mismo modelo, prompt de partida distinto.
inputs = processor(imagen, "a detailed photo showing", return_tensors="pt")
salida = blip_model.generate(**inputs, max_new_tokens=60)
summary = processor.decode(salida[0], skip_special_tokens=True)
print("Descripcion detallada:", summary)

Compara las dos descripciones: mismo modelo, mismo peso, y el único cambio es el texto de partida que le diste al generador. Es la misma idea que viste con el prompt de traducción en la clase anterior —el prompt determina el comportamiento del modelo— pero aquí en un contexto donde SÍ funciona como se espera, porque ambos prompts piden lo mismo (describir la imagen) con distinto nivel de detalle, en vez de pedir tareas completamente distintas (traducir vs. responder cualquier cosa).

## Cierre: todo lo que construiste en este curso

A lo largo de las 12 clases construiste, paso a paso:

1. **Forward propagation y funciones de activación** (clase 1) — cómo una red neuronal calcula una predicción, a mano con NumPy.
2. **Backpropagation** (clase 2) — cómo una red neuronal aprende, ajustando pesos con el problema XOR.
3. **Regresión y clasificación con Keras** (clase 3) — de NumPy puro a un framework de alto nivel, con datos reales (resistencia del concreto, dígitos MNIST).
4. **Redes convolucionales, CNN** (clase 4) — extracción de patrones visuales con `Conv2D` y `MaxPooling2D`.
5. **Self-attention** (clase 5) — la idea de Query/Key/Value detrás de los modelos de atención modernos.
6. **LLMs y chatbot con Hugging Face** (clase 6) — tu primer chatbot de terminal, con Blenderbot y SmolLM2.
7. **Chatbot web con Flask** (clase 7) — el mismo chatbot, ahora accesible desde un navegador.
8. **Whisper y reconocimiento de voz** (clase 8) — transcripción automática de audio, y resumen de reuniones.
9. **Asistente de voz conversacional** (clase 9) — el ciclo completo voz → LLM → voz.
10. **RAG sobre tus documentos** (clase 10) — un chatbot que responde con base en tus propios PDFs.
11. **Traducción con Babel Fish** (clase 11) — la misma arquitectura de voz, aplicada a traducción, y una lección real sobre verificar tus prompts.
12. **Visión y lenguaje: clasificación y captioning** (clase 12, esta clase) — transfer learning con VGG16 y descripciones de imágenes con BLIP.

Revisa el capítulo de cierre del manual, [17_casos_de_uso_reales.md](../manual/17_casos_de_uso_reales.md), para un repaso de los 7 casos de uso reales que construiste y ver por dónde seguir aprendiendo.

**Sigue con:**
- Capítulo del manual: [16_vision_lenguaje_proyecto_final.md](../manual/16_vision_lenguaje_proyecto_final.md)
- Notebook de práctica (capstone): [clase12_practica_vision_lenguaje_casos_uso.ipynb](../notebooks_practica/clase12_practica_vision_lenguaje_casos_uso.ipynb)